### Setting up LORA 

In [ ]:
q_proj = model.model.layers[0].self_attn.q_proj

print("Layer:", q_proj)
print("Weight shape:", tuple(q_proj.weight.shape))
print("Bias shape:", tuple(q_proj.bias.shape) if q_proj.bias is not None else None)
print("Trainable parameters:", sum(p.numel() for p in q_proj.parameters() if p.requires_grad))

In [ ]:
import torch
import torch.nn as nn

rank, alpha = 4, 4
A = nn.Parameter(torch.randn(rank, q_proj.in_features))
B = nn.Parameter(torch.zeros(q_proj.out_features, rank))

print(A.shape, B.shape, (B @ A).shape)

In [ ]:
x = torch.randn(1, 2048)
x.shape

In [ ]:
low_rank = x @ A.T
adjustment = low_rank @B.T

In [ ]:
print("Low-rank shape:", low_rank.shape)
print("Adjustment shape:", adjustment.shape)
print("Adjustment is zero:", torch.all(adjustment == 0).item())

In [ ]:
alpha =4 
x = x.to(device = q_proj.weight.device, dtype = q_proj.weight.dtype)

In [ ]:
base_output = q_proj(x)

In [ ]:
adjustment = (x @A.to(x).T) @ B.to(x).T
lora_output = base_output  + (alpha /rank) * adjustment

print(lora_output.shape)
print(torch.allclose(base_output, lora_output))

### Lora Linear Layer 

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer:nn.Linear, rank, alpha):
        super().__init__()
        self.base_layer = base_layer 
        self.base_layer.requires_grad_(False)
        
        self.A = nn.Linear(base_layer.in_features, rank, bias=False)
        self.B = nn.Linear(rank, base_layer.out_features, bias=False)
        
        nn.init.kaiming_uniform_(self.A.weight, a=5**0.5)
        nn.init.zeros_(self.B.weight)
        self.scaling = alpha / rank
        
    def forward(self, x):
        return self.base_layer(x) + self.scaling * self.B(self.A(x))
        
toy = nn.Linear(8, 6)
lora = LoRALinear(toy, rank, alpha)

print(lora.base_layer.weight.requires_grad)
print(lora.base_layer.bias.requires_grad)

In [ ]:
x = torch.randn(1, 8)
print(torch.allclose(lora(x), lora.base_layer(x)))

In [ ]:
lora.zero_grad()
lora(x).sum().backward()

print("Base:", lora.base_layer.weight.grad)
print("A gradient norm:", lora.A.weight.grad.norm().item())
print("B gradient norm:", lora.B.weight.grad.norm().item())

In [ ]:
optimizer = torch.optim.SGD([lora.A.weight, lora.B.weight], lr=0.01)

b_before = lora.B.weight.detach().clone()
optimizer.step()

print("B changed:", not torch.equal(b_before, lora.B.weight))
print("Output still matches base:", torch.allclose(lora(x), lora.base_layer(x)))

In [ ]:
print(model.model.layers[0])

In [ ]:
q_proj = model.model.layers[0].self_attn.q_proj
q_lora = LoRALinear(q_proj, rank=4, alpha=4).to(q_proj.weight)
print(q_lora)

In [ ]:
model.model.layers[0].self_attn.q_proj = q_lora

print(model.model.layers[0].self_attn.q_proj)

In [ ]:
model.requires_grad_(False)
q_lora.A.requires_grad_(True)
q_lora.B.requires_grad_(True)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

In [ ]:
attn = model.model.layers[0].self_attn
v_proj = attn.v_proj

attn.v_proj = LoRALinear(v_proj, rank=4, alpha=4).to(v_proj.weight)

print(attn.v_proj)